<a href="https://colab.research.google.com/github/Ca3ra/AIchatbot/blob/phase-3/AI3_Week3_retrieval_updated%20(ver2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BOI Investors' Guidebook RAG Chatbot: Week 3 (Retrieval), updated with Step 1 of "Next steps after Week 3"

This notebook is the Week 3 notebook with the Step 1 fixes added. Cells that changed are marked **[Step 1]**.

| Step 1 item | Where |
|---|---|
| Rewrite `requirements.txt` from a fixed list | Cell 11 |
| Rerun the 1500-character test with the BGE-M3 length cap raised | Cell 4 and Cell 10 |
| Grow the dev set to about 30 answerable questions (8-10 Tagalog/Taglish) | Cell 3a (you write the new questions) |
| Add 6-8 unanswerable questions, including near-miss ones | Cell 3a |
| Answer-bearing hit metric (`must_contain` found in a retrieved chunk) | Cell 5 |
| Rerun dense vs hybrid, optional lower BM25 weight and stopword removal | Cells 7, 7b, 8 |
| Refusal finding: does a top-1 score threshold separate answerable from unanswerable? | Cell 9 |
| Check the fee/days chunks without a column label | Cell 10 |

**Pipeline steps (unchanged):**
1. Write the development (dev) questions, each with a gold page, and check every gold page against the guidebook text.
2. Embed the chunks with MiniLM (English baseline) and two multilingual models, and save each FAISS index and the chunk metadata to Drive.
3. Measure hit@k, Precision@k, Recall@k and MRR@k for k = 1, 3, 5, separately for English and Tagalog/Taglish questions.
4. Add BM25 and compare dense only with hybrid (dense + BM25).
5. Check table chunks and test whether chunk size 1000 holds up.

**Two relevance definitions are now reported side by side:**
- **page-hit:** a retrieved chunk is relevant when its `pdf_pages` contains the question's gold page (the Week 3 definition).
- **answer-hit:** a retrieved chunk contains the question's `must_contain` phrase (after lowercasing, and normalising whitespace and hyphens). This is stricter: it shows whether the retrieved text can actually answer the question.

**Run order:** Cells run top to bottom. The embedding cell takes a few minutes the first time and is cached on Drive afterwards.


## Cell 1: Setup

In [1]:
!pip -q install faiss-cpu rank-bm25 sentence-transformers
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2: Config and load the Week 2 output

In [2]:
import json, re, csv, time, random, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

BASE = Path("/content/drive/MyDrive/AI3-Project")
PROCESSED = BASE / "data/processed"
INDEX_DIR = BASE / "index"
LOGS = BASE / "logs"
for p in (INDEX_DIR, LOGS):
    p.mkdir(parents=True, exist_ok=True)

K_VALUES = (1, 3, 5)
TOP_K_MAX = max(K_VALUES)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

chunks = read_jsonl(PROCESSED / "chunks.jsonl")
pages = read_jsonl(PROCESSED / "pages_clean.jsonl")
page_text = {r["pdf_page"]: " ".join(r["clean_text"].split()) for r in pages}   # line wraps -> single spaces

assert len({c["chunk_id"] for c in chunks}) == len(chunks), "duplicate chunk ids"
print(f"Loaded {len(chunks)} chunks and {len(pages)} pages")

# Fingerprint of the chunk set: cached indexes are only reused when the chunks have not changed
CHUNK_FP = hashlib.md5("".join(c["chunk_id"] + c["text"] for c in chunks).encode("utf-8")).hexdigest()
print("Chunk fingerprint:", CHUNK_FP[:8])

Loaded 1281 chunks and 502 pages
Chunk fingerprint: 6d02fd7b


## Cell 3: Development questions
The original 20 dev questions (18 answerable, 2 unanswerable) are kept below. **[Step 1]** Cell 3a, just before them, is where you add the new questions so the set reaches about 30 answerable and 6-8 unanswerable ones.

Every answerable question has a `gold_page` (PDF page) and a `must_contain` phrase, and the validation at the end of the cell checks that phrase against the page, so each expected answer can be traced to a page.

| Type | What it tests |
|---|---|
| `direct` | the question uses the guidebook's own words |
| `paraphrase` | the same fact in different words |
| `acronym` | the question is mostly acronyms (FAOM, CIR, BE) |
| `lookalike` | similar pages or tables exist (several fee tables, several FDAC procedures, PWP vs SWP) |
| `multipage` | **[Step 1]** the answer sits in a procedure that continues over several pages |
| `unanswerable` | the guidebook has no answer; used for the refusal analysis in Cell 9 |

`lang` is `en` or `tl` (Tagalog or Taglish). These are dev questions used for tuning; the held-out test questions are written later in a separate file and run once.


In [3]:
# ===== Cell 3a [Step 1]: YOU write the new dev questions here =====
# Goal: about 30 answerable questions in total, 8-10 of them Tagalog/Taglish, plus 6-8 unanswerable ones.
# Add: look-alike procedures, multi-page answers, paraphrases that do NOT copy the page's wording,
#      and near-miss unanswerable questions (topics the guide touches but does not answer).

def norm(s):
    """lowercase, hyphens/dashes -> space, collapse whitespace (used for must_contain checks and answer-hit)"""
    s = str(s).lower().replace("\u00ad", "")          # soft hyphen: delete (it only marks a possible line break)
    s = re.sub(r"[\-\u2010-\u2015]", " ", s)         # real hyphens and dashes -> space
    return " ".join(s.split())

def find(phrase, show=5):
    """Which PDF pages contain this phrase? Use it to find a gold_page and to prove an unanswerable question is really unanswerable."""
    hits = [p for p, t in page_text.items() if norm(phrase) in norm(t)]
    print(f"{phrase!r}: found on {len(hits)} page(s) {hits[:show]}{' ...' if len(hits) > show else ''}")
    return hits

# Answerable template (copy, fill in, delete the leading '#'):
# {"id": "D21", "lang": "en", "qtype": "paraphrase", "gold_page": 000, "must_contain": "exact phrase on that page",
#  "q": "Your question, in your own words", "expected": "The expected answer"},
#
# Multi-page answer: set gold_page to the page that holds the must_contain phrase; use qtype "multipage".
# Tagalog/Taglish: lang "tl". Keep official names and acronyms as written, e.g. "Magkano ang bayad para sa ...?"
#
# Unanswerable template (no gold page). near_miss=True when the guide touches the topic but does not answer it:
# {"id": "D31", "lang": "en", "qtype": "unanswerable", "near_miss": True, "gold_page": None, "must_contain": None,
#  "q": "...", "expected": "Not in the guidebook: the chatbot should say it does not know"},
#
# Before you add an unanswerable question, run find("a key word of the answer") and check it really is absent.

NEW_DEV_QUESTIONS = [
    # add your questions here
]


In [4]:
DEV_QUESTIONS = [
    # --- carried over from Week 1 (the first five dev questions) ---
    {"id": "D01", "lang": "en", "qtype": "direct", "gold_page": 465, "must_contain": "3,000",
     "q": "What is the BOI registration fee for a micro project?",
     "expected": "PhP 3,000"},
    {"id": "D02", "lang": "en", "qtype": "direct", "gold_page": 39, "must_contain": "at least 35 years old",
     "q": "What is the minimum age of a principal applicant for the Special Retiree's Resident Visa (SRRV)?",
     "expected": "At least 35 years old"},
    {"id": "D03", "lang": "en", "qtype": "direct", "gold_page": 68, "must_contain": "BIR Form No. 1902",
     "q": "Which BIR form is needed to apply online for a TIN for a local employee, and by when must the documents be submitted?",
     "expected": "BIR Form No. 1902 through eREG, before the 10th day of the following month"},
    {"id": "D04", "lang": "en", "qtype": "lookalike", "gold_page": 20, "must_contain": "three to six months",
     "q": "How long is the work covered by a Special Work Permit (SWP) - Commercial?",
     "expected": "Gainful employment of three to six months"},
    {"id": "D05", "lang": "en", "qtype": "direct", "gold_page": 454, "must_contain": "DOLE Regional Offices",
     "q": "Which office issues the Certificate of Registration of Job Contractors and Sub-contractors?",
     "expected": "DOLE Regional Offices (Articles 106 to 109 of the Labor Code)"},
    # --- new in Week 3 ---
    {"id": "D06", "lang": "en", "qtype": "paraphrase", "gold_page": 13, "must_contain": "Department of Labor and Employment",
     "q": "Which government department must approve a foreigner's permit before he or she can be employed in the Philippines?",
     "expected": "Department of Labor and Employment (DOLE), through the Alien Employment Permit"},
    {"id": "D07", "lang": "en", "qtype": "acronym", "gold_page": 115, "must_contain": "First Action on the Merits",
     "q": "What does FAOM stand for?",
     "expected": "First Action on the Merits (FAOM) Report"},
    {"id": "D08", "lang": "en", "qtype": "acronym", "gold_page": 101, "must_contain": "Certificate of Inward Remittance",
     "q": "What is the CIR for an FX inward remittance?",
     "expected": "Certificate of Inward Remittance of FX through an AAB, as proof of funding"},
    {"id": "D09", "lang": "en", "qtype": "lookalike", "gold_page": 119, "must_contain": "1,800",
     "q": "What is the priority claim fee for a big entity?",
     "expected": "Big Entity PhP 1,800 (Small Entity PhP 860)"},
    {"id": "D10", "lang": "en", "qtype": "lookalike", "gold_page": 115, "must_contain": "4,200",
     "q": "What is the substantive examination fee for a big entity?",
     "expected": "Big Entity PhP 4,200 (Small Entity PhP 2,010)"},
    {"id": "D11", "lang": "en", "qtype": "paraphrase", "gold_page": 126, "must_contain": "Industrial Design in 5 days",
     "q": "How fast is the express registration of an industrial design?",
     "expected": "\"Industrial Design in 5 days\" (Utility Model in 2 Months)"},
    {"id": "D12", "lang": "en", "qtype": "lookalike", "gold_page": 380, "must_contain": "Thursdays",
     "q": "On which day can applicants submit requirements to FDAC for pre-assessment?",
     "expected": "Thursdays, 8AM to 12NN (except holidays)"},
    {"id": "D13", "lang": "en", "qtype": "acronym", "gold_page": 300, "must_contain": "12 days",
     "q": "How many days does the desktop review of a BE inspection application take?",
     "expected": "12 days"},
    {"id": "D14", "lang": "en", "qtype": "direct", "gold_page": 80, "must_contain": "PhiFintech Innovation Office",
     "q": "Which SEC office gives clearance for FinTech-related business activities?",
     "expected": "PhiFintech Innovation Office (PIO)"},
    # --- Tagalog and Taglish ---
    {"id": "D15", "lang": "tl", "qtype": "paraphrase", "gold_page": 13, "must_contain": "Department of Labor and Employment",
     "q": "Anong ahensya ng gobyerno ang nagbibigay ng Alien Employment Permit (AEP)?",
     "expected": "Department of Labor and Employment (DOLE)"},
    {"id": "D16", "lang": "tl", "qtype": "direct", "gold_page": 141, "must_contain": "contact_us@dar.gov.ph",
     "q": "Ano ang email address ng DAR?",
     "expected": "contact_us@dar.gov.ph"},
    {"id": "D17", "lang": "tl", "qtype": "lookalike", "gold_page": 115, "must_contain": "4,200",
     "q": "Magkano ang Substantive Examination Fee para sa Big Entity?",
     "expected": "Big Entity PhP 4,200 (same fact as D10, asked in Taglish)"},
    {"id": "D18", "lang": "tl", "qtype": "direct", "gold_page": 500, "must_contain": "FONDEVILLA",
     "q": "Sino ang Executive Director ng BOI Investments Assistance Center?",
     "expected": "Atty. Bobby G. Fondevilla"},
    # --- unanswerable (no gold page) ---
    {"id": "D19", "lang": "en", "qtype": "unanswerable", "gold_page": None, "must_contain": None,
     "q": "What is the visa fee for Philippine citizens visiting Japan?",
     "expected": "Not in the guidebook: the chatbot should say it does not know"},
    {"id": "D20", "lang": "tl", "qtype": "unanswerable", "gold_page": None, "must_contain": None,
     "q": "Magkano ang presyo ng bigas ngayon sa palengke?",
     "expected": "Not in the guidebook: the chatbot should say it does not know"},
] + NEW_DEV_QUESTIONS

assert len({d["id"] for d in DEV_QUESTIONS}) == len(DEV_QUESTIONS), "duplicate question ids"

# 1) every expected answer must be on its gold page (case-insensitive; line wraps and hyphens ignored)
for d in DEV_QUESTIONS:
    if d["gold_page"] is None:
        continue
    assert d["gold_page"] in page_text, f"{d['id']}: PDF page {d['gold_page']} not found"
    assert norm(d["must_contain"]) in norm(page_text[d["gold_page"]]), \
        f"{d['id']}: '{d['must_contain']}' not found on PDF page {d['gold_page']}"

# 2) relevant chunks per question = chunks whose pdf_pages contains the gold page
answerable = [d for d in DEV_QUESTIONS if d["gold_page"] is not None]
unanswerable = [d for d in DEV_QUESTIONS if d["gold_page"] is None]
relevant = {d["id"]: {i for i, c in enumerate(chunks) if d["gold_page"] in c["pdf_pages"]} for d in answerable}
empty = [qid for qid, s in relevant.items() if not s]
assert not empty, f"no chunk covers the gold page for: {empty}"

print(f"{len(DEV_QUESTIONS)} dev questions: {len(answerable)} answerable, {len(unanswerable)} unanswerable")
print("By language:", dict(Counter(d["lang"] for d in DEV_QUESTIONS)))
print("By type:    ", dict(Counter(d["qtype"] for d in DEV_QUESTIONS)))
print("Relevant chunks per question:", {qid: len(s) for qid, s in relevant.items()})
print("All expected answers found on their gold pages.")

# [Step 1] How far are you from the Step 1 targets?
n_tl_ans = sum(d["lang"] == "tl" for d in answerable)
n_near = sum(bool(d.get("near_miss")) for d in unanswerable)
print("\nStep 1 targets (about):")
print(f"  answerable questions : {len(answerable):>2} / 30")
print(f"  Tagalog/Taglish (answerable) : {n_tl_ans:>2} / 8-10")
print(f"  unanswerable         : {len(unanswerable):>2} / 6-8   (near-miss: {n_near}, aim for at least 3)")
print(f"  look-alike: {sum(d['qtype']=='lookalike' for d in answerable)}, multipage: {sum(d['qtype']=='multipage' for d in answerable)}, paraphrase: {sum(d['qtype']=='paraphrase' for d in answerable)}")

20 dev questions: 18 answerable, 2 unanswerable
By language: {'en': 15, 'tl': 5}
By type:     {'direct': 7, 'lookalike': 5, 'paraphrase': 3, 'acronym': 3, 'unanswerable': 2}
Relevant chunks per question: {'D01': 1, 'D02': 4, 'D03': 3, 'D04': 4, 'D05': 4, 'D06': 4, 'D07': 2, 'D08': 6, 'D09': 3, 'D10': 2, 'D11': 4, 'D12': 4, 'D13': 4, 'D14': 5, 'D15': 4, 'D16': 1, 'D17': 2, 'D18': 2}
All expected answers found on their gold pages.

Step 1 targets (about):
  answerable questions : 18 / 30
  Tagalog/Taglish (answerable) :  4 / 8-10
  unanswerable         :  2 / 6-8   (near-miss: 0, aim for at least 3)
  look-alike: 5, multipage: 0, paraphrase: 3


## Cell 4: Embedding models
| Key | Model | Why |
|---|---|---|
| `minilm` | all-MiniLM-L6-v2 | English baseline from Week 2 |
| `e5_base` | multilingual-e5-base | small multilingual model; needs the `query: ` and `passage: ` prefixes |
| `bge_m3` | BGE-M3 | larger multilingual model with a long context; no prefixes |

The guidebook stays in English. The multilingual models are there so that a Tagalog question lands near the English chunk that answers it. To skip a model, remove its key from `MODELS_TO_RUN`. Check the exact model names on Hugging Face before you cite them.

**[Step 1]** The BGE-M3 length cap is raised from 512 to 1024 tokens so the 1500-character chunks in Cell 10 are not cut off before embedding. The 1000-character chunks are at most 378 tokens, so the cached 1000-character index does not change. `multilingual-e5-base` still truncates at 512 tokens (its limit); say so if you report e5 on longer chunks.

In [5]:
import faiss
from sentence_transformers import SentenceTransformer

MODELS = {
    "minilm": {"hf_id": "sentence-transformers/all-MiniLM-L6-v2", "q_prefix": "", "p_prefix": "", "max_len": None},
    "e5_base": {"hf_id": "intfloat/multilingual-e5-base", "q_prefix": "query: ", "p_prefix": "passage: ", "max_len": 512},
    "bge_m3": {"hf_id": "BAAI/bge-m3", "q_prefix": "", "p_prefix": "", "max_len": 1024},  # [Step 1] window is 8k tokens; 1024 covers the 1500-character chunk test (was 512)
}
MODELS_TO_RUN = ["minilm", "e5_base", "bge_m3"]
FORCE_REBUILD = False          # True = ignore cached indexes on Drive

def load_model(key):
    cfg = MODELS[key]
    m = SentenceTransformer(cfg["hf_id"])           # uses the GPU automatically when there is one
    if cfg["max_len"]:
        m.max_seq_length = min(m.max_seq_length, cfg["max_len"])
    return m

def passage_texts(key, chs):
    return [MODELS[key]["p_prefix"] + c["text"] for c in chs]

def token_stats(model, texts):
    lens = [len(model.tokenizer.encode(t, add_special_tokens=True)) for t in texts]
    limit = model.max_seq_length
    over = sum(1 for n in lens if n > limit)
    return {"max_seq_length": limit, "avg_tokens": round(sum(lens) / len(lens), 1), "max_tokens": max(lens),
            "over_limit": over, "over_limit_pct": round(100 * over / len(lens), 1)}

models, indexes, token_rows, embed_seconds = {}, {}, [], {}
for key in MODELS_TO_RUN:
    m = load_model(key)
    models[key] = m
    token_rows.append({"model": key, **token_stats(m, passage_texts(key, chunks))})

    idx_path = INDEX_DIR / f"{key}_{CHUNK_FP[:8]}.faiss"
    if idx_path.exists() and not FORCE_REBUILD:
        indexes[key] = faiss.read_index(str(idx_path))
        embed_seconds[key] = None
        print(f"{key}: loaded cached index {idx_path.name}")
    else:
        t0 = time.time()
        emb = m.encode(passage_texts(key, chunks), batch_size=32, normalize_embeddings=True,
                       show_progress_bar=True, convert_to_numpy=True).astype("float32")
        index = faiss.IndexFlatIP(emb.shape[1])        # exact search; vectors are normalised, so inner product = cosine
        index.add(emb)
        faiss.write_index(index, str(idx_path))
        indexes[key] = index
        embed_seconds[key] = round(time.time() - t0, 1)
        print(f"{key}: embedded {len(chunks)} chunks in {embed_seconds[key]}s, saved {idx_path.name}")
    assert indexes[key].ntotal == len(chunks), f"{key}: index size differs from the chunk count"

# Chunk metadata in index order (row i of every index = chunks[i])
with open(INDEX_DIR / "chunk_meta.json", "w", encoding="utf-8") as f:
    json.dump({"fingerprint": CHUNK_FP, "n_chunks": len(chunks), "models": {k: MODELS[k] for k in MODELS_TO_RUN},
               "chunks": chunks}, f, ensure_ascii=False)
print("Saved chunk_meta.json and", len(indexes), "index file(s) to", INDEX_DIR)

print("\nDoes every chunk fit the model's window? (text beyond max_seq_length is not embedded)")
display(pd.DataFrame(token_rows))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (37 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

bge_m3: loaded cached index bge_m3_6d02fd7b.faiss
Saved chunk_meta.json and 3 index file(s) to /content/drive/MyDrive/AI3-Project/index

Does every chunk fit the model's window? (text beyond max_seq_length is not embedded)


,model,max_seq_length,avg_tokens,max_tokens,over_limit,over_limit_pct
0,minilm,256,182.5,378,36,2.8
1,e5_base,512,217.1,393,0,0.0
2,bge_m3,1024,215.1,391,0,0.0


## Cell 5: Retrieval functions and metrics
- **hit@k:** 1 if at least one of the top k chunks is relevant.
- **Precision@k:** relevant chunks in the top k, divided by k.
- **Recall@k:** relevant chunks in the top k, divided by all relevant chunks for that question. A page is often split over several chunks, so Recall@k stays low even when retrieval works; read it together with hit@k.
- **MRR@k:** 1 / rank of the first relevant chunk (0 if none in the top k).

- **answer-hit@k [Step 1]:** 1 if the `must_contain` phrase appears in at least one of the top k chunks (whitespace and hyphens normalised first). Report it next to hit@k: a page can be retrieved without the chunk holding the answer.

Metrics use the answerable questions only. The unanswerable ones are used in Cell 9.

In [6]:
def dense_search(key, query, n=TOP_K_MAX):
    q = models[key].encode([MODELS[key]["q_prefix"] + query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    scores, ids = indexes[key].search(q, n)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]

def metrics_for(ranked, rel, k):
    hits = [1 if i in rel else 0 for i in ranked[:k]]
    n_rel = sum(hits)
    return {"hit": int(n_rel > 0), "precision": n_rel / k, "recall": n_rel / len(rel) if rel else 0.0,     # a chunk set may not cover the gold page at all
            "mrr": next((1 / (r + 1) for r, h in enumerate(hits) if h), 0.0)}

def answer_hit(ranked, d, k):
    """[Step 1] 1 if the must_contain phrase is in the text of one of the top-k chunks"""
    phrase = norm(d["must_contain"])
    return int(any(phrase in norm(chunks[i]["text"]) for i in ranked[:k]))

def evaluate(search_fn, label, questions=None):
    # search_fn(query, n) -> [(chunk_index, score), ...] best first
    rows = []
    for d in (questions or answerable):
        ranked = [i for i, _ in search_fn(d["q"], TOP_K_MAX)]
        for k in K_VALUES:
            rows.append({"system": label, "id": d["id"], "lang": d["lang"], "qtype": d["qtype"], "k": k,
                         **metrics_for(ranked, relevant[d["id"]], k), "ans_hit": answer_hit(ranked, d, k)})
    return pd.DataFrame(rows)

def summarize(df):
    both = pd.concat([df, df.assign(lang="all")])
    s = both.groupby(["system", "lang", "k"])[["hit", "ans_hit", "precision", "recall", "mrr"]].mean().round(3).reset_index()
    s["n_questions"] = both.groupby(["system", "lang", "k"])["id"].nunique().values
    return s

def hit_table(summary):
    # page-hit and answer-hit side by side
    parts = []
    for col, name in (("hit", "page-hit"), ("ans_hit", "answer-hit")):
        t = summary.pivot_table(index=["system", "lang"], columns="k", values=col)
        t.columns = [f"{name}@{k}" for k in t.columns]
        parts.append(t)
    return pd.concat(parts, axis=1)

all_results = []     # every evaluation is appended here and saved in Cell 11

## Cell 6: Dense retrieval, each model

In [7]:
dense_results = {}
for key in MODELS_TO_RUN:
    df = evaluate(lambda q, n, key=key: dense_search(key, q, n), f"dense:{key}")
    dense_results[key] = df
    all_results.append(df)

dense_summary = summarize(pd.concat(dense_results.values()))
display(Markdown("### hit@k by model and language"))
display(hit_table(dense_summary))
display(Markdown("### Full metrics (English, Tagalog and all questions)"))
display(dense_summary)

### hit@k by model and language

page-hit@1  page-hit@3  page-hit@5  answer-hit@1  \
system        lang                                                     
dense:bge_m3  all        0.833       0.944       1.000         0.833   
              en         0.786       0.929       1.000         0.786   
              tl         1.000       1.000       1.000         1.000   
dense:e5_base all        0.667       0.889       0.944         0.722   
              en         0.571       0.857       0.929         0.643   
              tl         1.000       1.000       1.000         1.000   
dense:minilm  all        0.556       0.722       0.833         0.611   
              en         0.500       0.714       0.786         0.571   
              tl         0.750       0.750       1.000         0.750   

                    answer-hit@3  answer-hit@5  
system        lang                              
dense:bge_m3  all          0.889         0.889  
              en           0.857         0.857  
              tl           1.000         1.000  
dense:e5_base all          0.889         0.944  
              en           0.857         0.929  
              tl           1.000         1.000  
dense:minilm  all          0.722         0.778  
              en           0.714         0.786  
              tl           0.750         0.750

### Full metrics (English, Tagalog and all questions)

,system,lang,k,hit,ans_hit,precision,recall,mrr,n_questions
0,dense:bge_m3,all,1,0.833,0.833,0.833,0.338,0.833,18
1,dense:bge_m3,all,3,0.944,0.889,0.500,0.544,0.889,18
2,dense:bge_m3,all,5,1.000,0.889,0.400,0.684,0.900,18
3,dense:bge_m3,en,1,0.786,0.786,0.786,0.274,0.786,14
4,dense:bge_m3,en,3,0.929,0.857,0.476,0.449,0.857,14
5,dense:bge_m3,en,5,1.000,0.857,0.414,0.630,0.871,14
6,dense:bge_m3,tl,1,1.000,1.000,1.000,0.562,1.000,4
7,dense:bge_m3,tl,3,1.000,1.000,0.583,0.875,1.000,4
8,dense:bge_m3,tl,5,1.000,1.000,0.350,0.875,1.000,4
9,dense:e5_base,all,1,0.667,0.722,0.667,0.245,0.667,18


## Cell 7: BM25 and hybrid retrieval
BM25 matches exact words, so it should help with acronyms and names (AEP, FAOM, CIR). It should help less for Tagalog questions on English text, because most of the words will not match. Hybrid search merges the dense ranking and the BM25 ranking with reciprocal rank fusion (RRF): each chunk scores `1 / (60 + rank)` in each list and the scores are added. **[Step 1]** `rrf_fuse` now takes optional weights so Cell 7b can try a lower BM25 weight; the default (1.0, 1.0) reproduces the Week 3 result.

In [8]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())      # keeps acronyms and numbers such as "4" and "200"

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bm25_search(query, n=TOP_K_MAX):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:n]
    return [(int(i), float(scores[i])) for i in order]

HYBRID_POOL = 20       # candidates taken from each list before fusion
RRF_K = 60

def rrf_fuse(ranked_lists, k=RRF_K, weights=None):
    weights = weights or [1.0] * len(ranked_lists)
    fused = defaultdict(float)
    for w, lst in zip(weights, ranked_lists):
        for rank, (i, _) in enumerate(lst):
            fused[i] += w / (k + rank + 1)
    return sorted(fused.items(), key=lambda x: -x[1])

def hybrid_search(key, query, n=TOP_K_MAX, bm25_weight=1.0):
    return rrf_fuse([dense_search(key, query, HYBRID_POOL), bm25_search(query, HYBRID_POOL)],
                    weights=[1.0, bm25_weight])[:n]

bm25_df = evaluate(bm25_search, "bm25")
hybrid_dfs = {key: evaluate(lambda q, n, key=key: hybrid_search(key, q, n), f"hybrid:{key}") for key in MODELS_TO_RUN}
all_results += [bm25_df] + list(hybrid_dfs.values())

compare = summarize(pd.concat([bm25_df] + list(dense_results.values()) + list(hybrid_dfs.values())))
display(Markdown("### hit@k: BM25 vs dense vs hybrid, by language"))
display(hit_table(compare))

# [Step 1] lookup used by Cell 8: system label -> search function
SYSTEM_FNS = {"bm25": bm25_search}
for key in MODELS_TO_RUN:
    SYSTEM_FNS[f"dense:{key}"] = lambda q, n, key=key: dense_search(key, q, n)
    SYSTEM_FNS[f"hybrid:{key}"] = lambda q, n, key=key: hybrid_search(key, q, n)

### hit@k: BM25 vs dense vs hybrid, by language

page-hit@1  page-hit@3  page-hit@5  answer-hit@1  \
system         lang                                                     
bm25           all        0.611       0.722       0.778         0.667   
               en         0.571       0.714       0.786         0.714   
               tl         0.750       0.750       0.750         0.500   
dense:bge_m3   all        0.833       0.944       1.000         0.833   
               en         0.786       0.929       1.000         0.786   
               tl         1.000       1.000       1.000         1.000   
dense:e5_base  all        0.667       0.889       0.944         0.722   
               en         0.571       0.857       0.929         0.643   
               tl         1.000       1.000       1.000         1.000   
dense:minilm   all        0.556       0.722       0.833         0.611   
               en         0.500       0.714       0.786         0.571   
               tl         0.750       0.750       1.000         0.750   
hybrid:bge_m3  all        0.778       1.000       1.000         0.833   
               en         0.714       1.000       1.000         0.786   
               tl         1.000       1.000       1.000         1.000   
hybrid:e5_base all        0.722       0.944       0.944         0.833   
               en         0.643       0.929       0.929         0.786   
               tl         1.000       1.000       1.000         1.000   
hybrid:minilm  all        0.667       0.833       0.944         0.722   
               en         0.571       0.786       0.929         0.714   
               tl         1.000       1.000       1.000         0.750   

                     answer-hit@3  answer-hit@5  
system         lang                              
bm25           all          0.833         0.889  
               en           0.857         0.929  
               tl           0.750         0.750  
dense:bge_m3   all          0.889         0.889  
               en           0.857         0.857  
               tl           1.000         1.000  
dense:e5_base  all          0.889         0.944  
               en           0.857         0.929  
               tl           1.000         1.000  
dense:minilm   all          0.722         0.778  
               en           0.714         0.786  
               tl           0.750         0.750  
hybrid:bge_m3  all          0.889         0.944  
               en           0.857         0.929  
               tl           1.000         1.000  
hybrid:e5_base all          0.944         1.000  
               en           0.929         1.000  
               tl           1.000         1.000  
hybrid:minilm  all          0.778         0.889  
               en           0.786         0.929  
               tl           0.750         0.750

## Cell 7b: [Step 1] Optional BM25 variants (lower BM25 weight, stopword removal)
These are extra experiments on the **dev** set. They are shown in their own table and are **not** used to pick `BEST` in Cell 8, because trying many variants on about 30 questions makes it easy to find a "winner" that is only luck. If a variant is better, check it again on the held-out test in Week 5 before claiming it. One question is about 3 points of hit@k, so a gap of one or two questions is a tie.

The stopword list is small and keeps short acronyms such as `be` (BE inspection) on purpose.

In [9]:
STOP = set("""a an the of to in on for and or is are was were by with as at from that this it its what which who how much many do does can
i my me ko ang ng sa na ay mga ba po ano sino saan kailan magkano paano para""".split())   # 'be' is left out: BE is an acronym in this guidebook

def tokenize_sw(text):
    return [w for w in tokenize(text) if w not in STOP]

bm25_sw = BM25Okapi([tokenize_sw(c["text"]) for c in chunks])

def bm25_sw_search(query, n=TOP_K_MAX):
    scores = bm25_sw.get_scores(tokenize_sw(query))
    order = np.argsort(-scores)[:n]
    return [(int(i), float(scores[i])) for i in order]

def hybrid_variant(key, bm25_fn, w):
    return lambda q, n: rrf_fuse([dense_search(key, q, HYBRID_POOL), bm25_fn(q, HYBRID_POOL)], weights=[1.0, w])[:n]

variant_dfs = []
variant_dfs.append(evaluate(bm25_sw_search, "bm25+stopwords"))
SYSTEM_FNS["bm25+stopwords"] = bm25_sw_search
for key in MODELS_TO_RUN:
    for label, fn_bm25, w in (("bm25w0.5", bm25_search, 0.5), ("bm25w0.3", bm25_search, 0.3),
                              ("sw+bm25w1.0", bm25_sw_search, 1.0), ("sw+bm25w0.5", bm25_sw_search, 0.5)):
        name = f"hybrid:{key}@{label}"
        SYSTEM_FNS[name] = hybrid_variant(key, fn_bm25, w)
        variant_dfs.append(evaluate(SYSTEM_FNS[name], name))
all_results += variant_dfs

variants_summary = summarize(pd.concat(variant_dfs))
display(Markdown("### Variants: page-hit and answer-hit by language"))
display(hit_table(variants_summary))
display(Markdown("For comparison, the base systems from Cell 7:"))
display(hit_table(compare[compare["system"].str.startswith(("hybrid", "dense", "bm25"))]))

### Variants: page-hit and answer-hit by language

page-hit@1  page-hit@3  page-hit@5  \
system                     lang                                       
bm25+stopwords             all        0.611       0.722       0.833   
                           en         0.571       0.714       0.857   
                           tl         0.750       0.750       0.750   
hybrid:bge_m3@bm25w0.3     all        0.944       1.000       1.000   
                           en         0.929       1.000       1.000   
                           tl         1.000       1.000       1.000   
hybrid:bge_m3@bm25w0.5     all        0.944       1.000       1.000   
                           en         0.929       1.000       1.000   
                           tl         1.000       1.000       1.000   
hybrid:bge_m3@sw+bm25w0.5  all        0.889       1.000       1.000   
                           en         0.857       1.000       1.000   
                           tl         1.000       1.000       1.000   
hybrid:bge_m3@sw+bm25w1.0  all        0.778       1.000       1.000   
                           en         0.714       1.000       1.000   
                           tl         1.000       1.000       1.000   
hybrid:e5_base@bm25w0.3    all        0.722       0.944       0.944   
                           en         0.643       0.929       0.929   
                           tl         1.000       1.000       1.000   
hybrid:e5_base@bm25w0.5    all        0.722       0.944       0.944   
                           en         0.643       0.929       0.929   
                           tl         1.000       1.000       1.000   
hybrid:e5_base@sw+bm25w0.5 all        0.778       0.944       0.944   
                           en         0.714       0.929       0.929   
                           tl         1.000       1.000       1.000   
hybrid:e5_base@sw+bm25w1.0 all        0.722       0.944       0.944   
                           en         0.643       0.929       0.929   
                           tl         1.000       1.000       1.000   
hybrid:minilm@bm25w0.3     all        0.722       0.833       0.833   
                           en         0.643       0.786       0.786   
                           tl         1.000       1.000       1.000   
hybrid:minilm@bm25w0.5     all        0.667       0.833       0.833   
                           en         0.571       0.786       0.786   
                           tl         1.000       1.000       1.000   
hybrid:minilm@sw+bm25w0.5  all        0.667       0.833       0.833   
                           en         0.571       0.786       0.786   
                           tl         1.000       1.000       1.000   
hybrid:minilm@sw+bm25w1.0  all        0.722       0.778       0.889   
                           en         0.643       0.714       0.857   
                           tl         1.000       1.000       1.000   

                                 answer-hit@1  answer-hit@3  answer-hit@5  
system                     lang                                            
bm25+stopwords             all          0.556         0.778         0.944  
                           en           0.571         0.786         1.000  
                           tl           0.500         0.750         0.750  
hybrid:bge_m3@bm25w0.3     all          0.833         0.889         0.889  
                           en           0.786         0.857         0.857  
                           tl           1.000         1.000         1.000  
hybrid:bge_m3@bm25w0.5     all          0.833         0.889         0.889  
                           en           0.786         0.857         0.857  
                           tl           1.000         1.000         1.000  
hybrid:bge_m3@sw+bm25w0.5  all          0.778         0.889         0.944  
                           en           0.714         0.857         0.929  
                           tl           1.000         1.000         1.000  
hybrid:bge_m3@sw+bm25w1.0  all          0.778         0.8

For comparison, the base systems from Cell 7:

page-hit@1  page-hit@3  page-hit@5  answer-hit@1  \
system         lang                                                     
bm25           all        0.611       0.722       0.778         0.667   
               en         0.571       0.714       0.786         0.714   
               tl         0.750       0.750       0.750         0.500   
dense:bge_m3   all        0.833       0.944       1.000         0.833   
               en         0.786       0.929       1.000         0.786   
               tl         1.000       1.000       1.000         1.000   
dense:e5_base  all        0.667       0.889       0.944         0.722   
               en         0.571       0.857       0.929         0.643   
               tl         1.000       1.000       1.000         1.000   
dense:minilm   all        0.556       0.722       0.833         0.611   
               en         0.500       0.714       0.786         0.571   
               tl         0.750       0.750       1.000         0.750   
hybrid:bge_m3  all        0.778       1.000       1.000         0.833   
               en         0.714       1.000       1.000         0.786   
               tl         1.000       1.000       1.000         1.000   
hybrid:e5_base all        0.722       0.944       0.944         0.833   
               en         0.643       0.929       0.929         0.786   
               tl         1.000       1.000       1.000         1.000   
hybrid:minilm  all        0.667       0.833       0.944         0.722   
               en         0.571       0.786       0.929         0.714   
               tl         1.000       1.000       1.000         0.750   

                     answer-hit@3  answer-hit@5  
system         lang                              
bm25           all          0.833         0.889  
               en           0.857         0.929  
               tl           0.750         0.750  
dense:bge_m3   all          0.889         0.889  
               en           0.857         0.857  
               tl           1.000         1.000  
dense:e5_base  all          0.889         0.944  
               en           0.857         0.929  
               tl           1.000         1.000  
dense:minilm   all          0.722         0.778  
               en           0.714         0.786  
               tl           0.750         0.750  
hybrid:bge_m3  all          0.889         0.944  
               en           0.857         0.929  
               tl           1.000         1.000  
hybrid:e5_base all          0.944         1.000  
               en           0.929         1.000  
               tl           1.000         1.000  
hybrid:minilm  all          0.778         0.889  
               en           0.786         0.929  
               tl           0.750         0.750

## Cell 8: Which setup is best, and where does it fail?
**[Step 1]** The best system is now chosen by answer-hit@5 first, then page-hit@5, then MRR@5, and any system within one question of the best is listed as a tie. Report a tie as a tie.

In [21]:
# Best system = highest answer-hit@5, then page-hit@5, then MRR@5 (base systems from Cell 7 only)
cmp5 = compare[(compare["lang"] == "all") & (compare["k"] == TOP_K_MAX)].sort_values(["ans_hit", "hit", "mrr"], ascending=False)
display(cmp5[["system", "hit", "ans_hit", "precision", "recall", "mrr"]].reset_index(drop=True))
BEST = cmp5.iloc[0]["system"]
print("Best system on these dev questions:", BEST)

one_q = 1 / len(answerable)          # one question moves a hit rate by this much
top = cmp5.iloc[0]
TOL = one_q + 0.002                  # the table is rounded to 3 decimals, so 1.000 - 0.944 = 0.056 must still count as one question (0.0556)
ties = cmp5[(top["ans_hit"] - cmp5["ans_hit"] <= TOL) & (top["hit"] - cmp5["hit"] <= TOL)]
print(f"One question = {100*one_q:.1f} points. Systems within one question of the best on both metrics: {list(ties['system'])}")
if len(ties) > 1:
    print("-> Report this as a tie, not a win, and say why you still picked BEST (for example it is simpler, or better on Tagalog).")

def _search_fn(system):
    return SYSTEM_FNS[system]

fn = _search_fn(BEST)
misses = []
for d in answerable:
    ranked = fn(d["q"], TOP_K_MAX)
    got = [i for i, _ in ranked]
    if not (set(got) & relevant[d["id"]]):
        misses.append({"id": d["id"], "lang": d["lang"], "qtype": d["qtype"], "gold_page": d["gold_page"],
                       "question": d["q"][:70], "top1_pages": chunks[got[0]]["pdf_pages"], "top1_procedure": chunks[got[0]]["procedure"][:50]})
print(f"\nQuestions with no relevant chunk in the top {TOP_K_MAX} ({len(misses)} of {len(answerable)}):")
display(pd.DataFrame(misses) if misses else "none")

# Same comparison by question type, for the best system
by_type = pd.concat([df for df in all_results if df["system"].iloc[0] == BEST])
display(Markdown(f"### {BEST}: hit@{TOP_K_MAX} by question type"))
display(by_type[by_type["k"] == TOP_K_MAX].groupby("qtype")["hit"].agg(["mean", "count"]).round(2))

,system,hit,ans_hit,precision,recall,mrr
0,hybrid:e5_base,0.944,1.000,0.378,0.670,0.824
1,hybrid:bge_m3,1.000,0.944,0.344,0.613,0.889
2,dense:e5_base,0.944,0.944,0.356,0.596,0.792
3,dense:bge_m3,1.000,0.889,0.400,0.684,0.900
4,hybrid:minilm,0.944,0.889,0.344,0.585,0.756
5,bm25,0.778,0.889,0.300,0.506,0.681
6,dense:minilm,0.833,0.778,0.289,0.483,0.655


Best system on these dev questions: hybrid:e5_base
One question = 5.6 points. Systems within one question of the best on both metrics: ['hybrid:e5_base', 'hybrid:bge_m3', 'dense:e5_base']
-> Report this as a tie, not a win, and say why you still picked BEST (for example it is simpler, or better on Tagalog).

Questions with no relevant chunk in the top 5 (1 of 18):


,id,lang,qtype,gold_page,question,top1_pages,top1_procedure
0,D12,en,lookalike,380,On which day can applicants submit requirement...,"[354, 355]",Certificate of Product Registration (CPR) of P...


### hybrid:e5_base: hit@5 by question type

,mean,count
qtype,,
acronym,1.0,3
direct,1.0,7
lookalike,0.8,5
paraphrase,1.0,3


## Cell 9: Unanswerable questions and the refusal finding [Step 1]
The chatbot has to refuse when the answer is not in the guidebook. A single rule such as "refuse if the top-1 similarity is below t" only works if the top-1 scores of answerable and unanswerable questions separate. This cell checks that for each model:

- the range of top-1 scores for answerable and for unanswerable questions, and how many unanswerable questions score above the **lowest** answerable one (overlap);
- the best single threshold on this dev set, and how many questions it gets wrong. The threshold is picked on the same questions it is scored on, so its accuracy is optimistic;
- the near-miss questions separately, because they are the ones most likely to score high.

If the two groups overlap, the finding is: *a score threshold alone does not separate them, so refusal also needs a prompt rule* (Week 4). Write that finding in your own words, with your own numbers, in the last cell of this notebook.


In [22]:
rows, detail = [], {}
for key in MODELS_TO_RUN:
    ans = np.array([dense_search(key, d["q"], 1)[0][1] for d in answerable])
    una_scores = {d["id"]: dense_search(key, d["q"], 1)[0][1] for d in unanswerable}
    una = np.array(list(una_scores.values()))
    # best single threshold t: answer when top-1 >= t
    cands = np.unique(np.concatenate([ans, una]))
    acc, t = max(((((ans >= t).sum() + (una < t).sum()) / (len(ans) + len(una))), t) for t in cands)
    rows.append({"model": key,
                 "ans min": round(ans.min(), 3), "ans median": round(float(np.median(ans)), 3),
                 "unans median": round(float(np.median(una)), 3), "unans max": round(una.max(), 3),
                 "unans >= lowest ans": f"{int((una >= ans.min()).sum())} of {len(una)}",
                 "best t": round(float(t), 3),
                 "wrongly refused (answerable)": int((ans < t).sum()),
                 "wrongly answered (unanswerable)": int((una >= t).sum()),
                 "accuracy at best t": round(float(acc), 3)})
    detail[key] = (ans, una_scores)
display(pd.DataFrame(rows))

det = pd.DataFrame([{"id": d["id"], "lang": d["lang"], "near_miss": bool(d.get("near_miss")), "question": d["q"][:60],
                     **{key: round(detail[key][1][d["id"]], 3) for key in MODELS_TO_RUN},
                     **{f"{key} #answerable below": int((detail[key][0] < detail[key][1][d["id"]]).sum()) for key in MODELS_TO_RUN}}
                    for d in unanswerable])
display(Markdown("### Top-1 score of each unanswerable question (and how many answerable questions score lower)"))
display(det)
print("Scores are cosine similarities from each model, so compare them within a model, not across models.")
for r in rows:
    sep = r["unans max"] < r["ans min"]
    print(f"{r['model']}: groups {'DO' if sep else 'do NOT'} separate (unanswerable max {r['unans max']} vs answerable min {r['ans min']}).")
if len(unanswerable) < 6:
    print(f"Only {len(unanswerable)} unanswerable questions so far: add more in Cell 3a before drawing a conclusion.")

,model,ans min,ans median,unans median,unans max,unans >= lowest ans,best t,wrongly refused (answerable),wrongly answered (unanswerable),accuracy at best t
0,minilm,0.327,0.648,0.448,0.535,2 of 2,0.438,1,1,0.90
1,e5_base,0.780,0.864,0.800,0.832,1 of 2,0.834,1,0,0.95
2,bge_m3,0.498,0.664,0.502,0.570,1 of 2,0.571,1,0,0.95


### Top-1 score of each unanswerable question (and how many answerable questions score lower)

,id,lang,near_miss,question,minilm,e5_base,bge_m3,minilm #answerable below,e5_base #answerable below,bge_m3 #answerable below
0,D19,en,False,What is the visa fee for Philippine citizens v...,0.535,0.832,0.570,4,1,1
1,D20,tl,False,Magkano ang presyo ng bigas ngayon sa palengke?,0.361,0.768,0.433,1,0,0


Scores are cosine similarities from each model, so compare them within a model, not across models.
minilm: groups do NOT separate (unanswerable max 0.535 vs answerable min 0.327).
e5_base: groups do NOT separate (unanswerable max 0.832 vs answerable min 0.78).
bge_m3: groups do NOT separate (unanswerable max 0.57 vs answerable min 0.498).
Only 2 unanswerable questions so far: add more in Cell 3a before drawing a conclusion.


## Cell 10: Table chunks and chunk size
**Table chunks.** A fee or processing-time cell is only useful if the chunk also carries its column label. Week 2 does not repeat a table's header row in every chunk, so later chunks of a table may show numbers without labels. The cell below measures how often that happens and prints a few examples to read.

In [23]:
HEADER_WORDS = re.compile(r"\b(fees?|processing time|agency actions?|client|step|requirements?)\b", re.IGNORECASE)
table_chunks = [c for c in chunks if c.get("has_table")]
money_or_days = [c for c in table_chunks if re.search(r"₱|\bPhP\b|\bdays?\b", c["text"])]
labelled = [c for c in money_or_days if HEADER_WORDS.search(c["text"][c["breadcrumb_len"]:])]
print(f"{len(table_chunks)} table chunks; {len(money_or_days)} contain a fee or a number of days;")
print(f"{len(labelled)} of those ({100 * len(labelled) / max(1, len(money_or_days)):.0f}%) also contain a column label such as FEES, PROCESSING TIME or STEP.")

rng = random.Random(SEED)
unlabelled = [c for c in money_or_days if c not in labelled]
print(f"\n{len(unlabelled)} fee/days chunks without a column label. Five to read:")
for c in rng.sample(unlabelled, k=min(5, len(unlabelled))):
    print(f"\n--- {c['chunk_id']} pages={c['pdf_pages']} ---\n{c['text'][:400]}")

653 table chunks; 369 contain a fee or a number of days;
318 of those (86%) also contain a column label such as FEES, PROCESSING TIME or STEP.

51 fee/days chunks without a column label. Five to read:

--- w2_c0860 pages=[345] ---
[Center for Drug Regulation Research > Certificate of Product Registration (CPR) of Pharmaceutical Products (Initial...] Reviews the evaluated application bearing the recommendation of the Junior Evaluator | 12 days Prepares the final output document (CPR/LOD), affixes initial, and forwards it to the senior evaluator (FDRO III) If with post-approval commitment/s, prepares a letter, signs, and forw

--- w2_c0373 pages=[156, 157] ---
[Energy Resources Development Bureau > Acknowledgment of Administrative Subcontracts, Letters of Registrations, Defi...] Technical Evaluation | 8 days Evaluate subcontracts in accordance to the submitted Work Program and Budget for the Calendar Year for Petroleum Operations related subcontracts and/or Administrative subcontracts No

In [24]:
# [Step 1] Do the fee/days chunks without a column label hurt any dev question?
labelled_ids = {c["chunk_id"] for c in labelled}
money_ids = {c["chunk_id"] for c in money_or_days}
ref_key = "bge_m3" if "bge_m3" in dense_results else MODELS_TO_RUN[-1]
d5 = dense_results[ref_key]
d5 = d5[d5["k"] == TOP_K_MAX].set_index("id")

rows = []
for d in answerable:
    ph = norm(d["must_contain"])
    fee_chunks = [c for c in chunks if ph in norm(c["text"]) and d["gold_page"] in c["pdf_pages"] and c["chunk_id"] in money_ids]
    if fee_chunks:
        rows.append({"id": d["id"], "lang": d["lang"], "question": d["q"][:55],
                     "answer chunks (fee/days)": len(fee_chunks),
                     "without column label": sum(c["chunk_id"] not in labelled_ids for c in fee_chunks),
                     f"{ref_key} page-hit@{TOP_K_MAX}": int(d5.loc[d["id"], "hit"]),
                     f"{ref_key} answer-hit@{TOP_K_MAX}": int(d5.loc[d["id"], "ans_hit"])})
fee_df = pd.DataFrame(rows)
display(fee_df if len(fee_df) else "no fee/days questions found")
if len(fee_df):
    bad = fee_df[(fee_df["without column label"] > 0) & (fee_df[f"{ref_key} answer-hit@{TOP_K_MAX}"] == 0)]
    print(f"Fee/days questions whose answer chunk has no column label AND were not retrieved: {len(bad)}")
    print("If this is 0, the missing column labels did not hurt any dev question; note that and move on. If not, read those chunks (Week 2 table-header repeat is the likely fix).")

,id,lang,question,answer chunks (fee/days),without column label,bge_m3 page-hit@5,bge_m3 answer-hit@5
0,D01,en,What is the BOI registration fee for a micro p...,1,0,1,1
1,D06,en,Which government department must approve a for...,1,0,1,1
2,D07,en,What does FAOM stand for?,1,0,1,0
3,D09,en,What is the priority claim fee for a big entity?,1,0,1,1
4,D10,en,What is the substantive examination fee for a ...,1,0,1,1
5,D11,en,How fast is the express registration of an ind...,1,0,1,1
6,D13,en,How many days does the desktop review of a BE ...,2,0,1,1
7,D15,tl,Anong ahensya ng gobyerno ang nagbibigay ng Al...,1,0,1,1
8,D17,tl,Magkano ang Substantive Examination Fee para s...,1,0,1,1


Fee/days questions whose answer chunk has no column label AND were not retrieved: 0
If this is 0, the missing column labels did not hurt any dev question; note that and move on. If not, read those chunks (Week 2 table-header repeat is the likely fix).


**Chunk size.** Week 2 chose 1000 characters with a 150-character overlap, marked provisional. Cell W2-6b saves the four sizes that were compared (600, 800, 1000, 1500) to `data/processed/experiments/`. Here each size is embedded with the best dense model and scored with the same questions and the same relevance rule. The overlap is 150 characters at every size (25% of 600, 10% of 1500), so the comparison is not perfectly fair; treat small differences with caution.

**[Step 1]** The test now always uses BGE-M3 with the raised 1024-token cap, prints how many chunks of each size are still over the model's limit (should be 0), and reports answer-hit next to page-hit.

In [25]:
EXP_DIR = PROCESSED / "experiments"
BEST_DENSE = max(MODELS_TO_RUN, key=lambda k: dense_summary[(dense_summary["system"] == f"dense:{k}") & (dense_summary["lang"] == "all")
                                                         & (dense_summary["k"] == TOP_K_MAX)]["hit"].iloc[0])
CHUNK_MODEL = "bge_m3" if "bge_m3" in MODELS_TO_RUN else BEST_DENSE      # [Step 1] test with the model whose cap was raised
print("Best dense model on page-hit@5 (Cell 6):", BEST_DENSE, "| chunk-size test uses:", CHUNK_MODEL,
      "| max_seq_length =", models[CHUNK_MODEL].max_seq_length)

def evaluate_chunk_set(key, chs):
    # embed this chunk set and score it with the answerable dev questions
    emb = models[key].encode(passage_texts(key, chs), batch_size=32, normalize_embeddings=True,
                             show_progress_bar=False, convert_to_numpy=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
    rel = {d["id"]: {i for i, c in enumerate(chs) if d["gold_page"] in c["pdf_pages"]} for d in answerable}
    norm_texts = [norm(c["text"]) for c in chs]
    rows = []
    for d in answerable:
        q = models[key].encode([MODELS[key]["q_prefix"] + d["q"]], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        _, ids = index.search(q, TOP_K_MAX)
        ranked = [int(i) for i in ids[0] if i >= 0]
        ph = norm(d["must_contain"])
        for k in K_VALUES:
            rows.append({"lang": d["lang"], "k": k, **metrics_for(ranked, rel[d["id"]], k),
                         "ans_hit": int(any(ph in norm_texts[i] for i in ranked[:k]))})
    df = pd.DataFrame(rows)
    return df.groupby("k")[["hit", "ans_hit", "precision", "mrr"]].mean().round(3)

files = sorted(EXP_DIR.glob("chunks_*.jsonl"), key=lambda p: int(p.stem.split("_")[1])) if EXP_DIR.exists() else []
if not files:
    print("No chunk-size files found. Run Cell W2-6b in the Week 2 notebook first.")
else:
    size_rows = []
    for p in files:
        size = int(p.stem.split("_")[1])
        chs = read_jsonl(p)
        ts = token_stats(models[CHUNK_MODEL], passage_texts(CHUNK_MODEL, chs))      # [Step 1] is anything still cut off?
        res = evaluate_chunk_set(CHUNK_MODEL, chs)
        for k in K_VALUES:
            size_rows.append({"max_chars": size, "k": k, "chunks": len(chs), "max_tokens": ts["max_tokens"],
                              "over_limit": ts["over_limit"], **res.loc[k].to_dict()})
    size_df = pd.DataFrame(size_rows)
    print("Dense model used:", CHUNK_MODEL)
    display(size_df[size_df["k"] == 1][["max_chars", "chunks", "max_tokens", "over_limit"]].reset_index(drop=True))
    display(size_df.pivot_table(index=["max_chars", "chunks"], columns="k", values="hit").rename(columns=lambda k: f"page-hit@{k}"))
    display(size_df.pivot_table(index=["max_chars", "chunks"], columns="k", values="ans_hit").rename(columns=lambda k: f"answer-hit@{k}"))
    if size_df["over_limit"].max() > 0:
        print("WARNING: some chunks are still longer than the model window, so their end is not embedded. Raise max_len in Cell 4.")

Best dense model on page-hit@5 (Cell 6): bge_m3 | chunk-size test uses: bge_m3 | max_seq_length = 1024
Dense model used: bge_m3


,max_chars,chunks,max_tokens,over_limit
0,600,2474,237,0
1,800,1664,331,0
2,1000,1281,391,0
3,1500,801,585,0


,k,page-hit@1,page-hit@3,page-hit@5
max_chars,chunks,,,
600,2474,0.833,1.000,1.000
800,1664,0.778,0.944,0.944
1000,1281,0.833,0.944,1.000
1500,801,0.722,0.889,0.889


,k,answer-hit@1,answer-hit@3,answer-hit@5
max_chars,chunks,,,
600,2474,0.778,0.833,0.889
800,1664,0.667,0.889,0.889
1000,1281,0.833,0.889,0.889
1500,801,0.833,0.944,0.944


## Cell 11: Save results and write the summary

In [26]:
results_df = pd.concat(all_results, ignore_index=True)
results_path = LOGS / "week3_results.csv"
results_df.to_csv(results_path, index=False)
print("Saved per-question results to", results_path)

with open(INDEX_DIR / "dev_questions.json", "w", encoding="utf-8") as f:
    json.dump(DEV_QUESTIONS, f, ensure_ascii=False, indent=1)

# [Step 1] requirements.txt from a fixed list. (The old pip freeze | grep only kept packages installed in this session, so it wrote 3 of 6.)
!pip -q install pypdf pymupdf pdfplumber "pillow<13"
import importlib.metadata as md
REQS = ["pypdf", "pymupdf", "pdfplumber", "sentence-transformers", "faiss-cpu", "rank-bm25", "pandas", "numpy"]
req_lines = []
for name in REQS:
    try:
        req_lines.append(f"{name}=={md.version(name)}")
    except md.PackageNotFoundError:
        req_lines.append(name)          # not installed here: listed without a version
(BASE / "requirements.txt").write_text("\n".join(req_lines) + "\n")
assert len(req_lines) == len(REQS)
print((BASE / "requirements.txt").read_text())
# Week 4 will add: transformers, accelerate, bitsandbytes, gradio

Saved per-question results to /content/drive/MyDrive/AI3-Project/logs/week3_results.csv
pypdf==7.1.1
pymupdf==2.1
pdfplumber==0.11.10
sentence-transformers==5.7.0
faiss-cpu==1.16.0
rank-bm25==0.2.2
pandas==2.2.3
numpy==2.4.6



In [27]:
def _row(system, lang, k):
    r = compare[(compare["system"] == system) & (compare["lang"] == lang) & (compare["k"] == k)]
    return r.iloc[0] if len(r) else None

lines = ["| system | EN page-hit @1 / @3 / @5 | TL page-hit @1 / @3 / @5 | EN answer-hit @5 | TL answer-hit @5 | all MRR@5 |", "|---|---|---|---|---|---|"]
for system in compare["system"].unique():
    en = " / ".join(f"{_row(system, 'en', k)['hit']:.2f}" for k in K_VALUES)
    tl = " / ".join(f"{_row(system, 'tl', k)['hit']:.2f}" for k in K_VALUES)
    en_a = f"{_row(system, 'en', TOP_K_MAX)['ans_hit']:.2f}"
    tl_a = f"{_row(system, 'tl', TOP_K_MAX)['ans_hit']:.2f}"
    lines.append(f"| {system} | {en} | {tl} | {en_a} | {tl_a} | {_row(system, 'all', TOP_K_MAX)['mrr']:.2f} |")

n_en = sum(d["lang"] == "en" for d in answerable)
n_tl = sum(d["lang"] == "tl" for d in answerable)
display(Markdown(f'''## Week 3 summary (after Step 1)
- **Index:** {len(chunks)} chunks (max {max(c["char_len"] for c in chunks)} characters), exact FAISS `IndexFlatIP` per model, saved with the chunk metadata to `{INDEX_DIR.name}/` on Drive.
- **Questions:** {len(DEV_QUESTIONS)} dev questions ({len(answerable)} answerable: {n_en} English, {n_tl} Tagalog/Taglish; {len(unanswerable)} unanswerable, {sum(bool(d.get("near_miss")) for d in unanswerable)} near-miss). page-hit = the chunk's `pdf_pages` contains the gold page; answer-hit = a retrieved chunk contains the `must_contain` phrase.
- **Best system on these dev questions:** `{BEST}` (see the tie check in Cell 8; report a tie as a tie).

{chr(10).join(lines)}

**Limits to state in the report:** the dev set is small ({len(answerable)} answerable questions), so one question moves hit@k by about {100 / len(answerable):.0f} points and the Tagalog numbers rest on only {n_tl} questions. Chunk size 1000 stays provisional until the held-out test questions confirm it. Recall@k is low by design because a page is split over several chunks.
'''))

## Week 3 summary (after Step 1)
- **Index:** 1281 chunks (max 1000 characters), exact FAISS `IndexFlatIP` per model, saved with the chunk metadata to `index/` on Drive.
- **Questions:** 20 dev questions (18 answerable: 14 English, 4 Tagalog/Taglish; 2 unanswerable, 0 near-miss). page-hit = the chunk's `pdf_pages` contains the gold page; answer-hit = a retrieved chunk contains the `must_contain` phrase.
- **Best system on these dev questions:** `hybrid:e5_base` (see the tie check in Cell 8; report a tie as a tie).

| system | EN page-hit @1 / @3 / @5 | TL page-hit @1 / @3 / @5 | EN answer-hit @5 | TL answer-hit @5 | all MRR@5 |
|---|---|---|---|---|---|
| bm25 | 0.57 / 0.71 / 0.79 | 0.75 / 0.75 / 0.75 | 0.93 | 0.75 | 0.68 |
| dense:bge_m3 | 0.79 / 0.93 / 1.00 | 1.00 / 1.00 / 1.00 | 0.86 | 1.00 | 0.90 |
| dense:e5_base | 0.57 / 0.86 / 0.93 | 1.00 / 1.00 / 1.00 | 0.93 | 1.00 | 0.79 |
| dense:minilm | 0.50 / 0.71 / 0.79 | 0.75 / 0.75 / 1.00 | 0.79 | 0.75 | 0.66 |
| hybrid:bge_m3 | 0.71 / 1.00 / 1.00 | 1.00 / 1.00 / 1.00 | 0.93 | 1.00 | 0.89 |
| hybrid:e5_base | 0.64 / 0.93 / 0.93 | 1.00 / 1.00 / 1.00 | 1.00 | 1.00 | 0.82 |
| hybrid:minilm | 0.57 / 0.79 / 0.93 | 1.00 / 1.00 / 1.00 | 0.93 | 0.75 | 0.76 |

**Limits to state in the report:** the dev set is small (18 answerable questions), so one question moves hit@k by about 6 points and the Tagalog numbers rest on only 4 questions. Chunk size 1000 stays provisional until the held-out test questions confirm it. Recall@k is low by design because a page is split over several chunks.


In [28]:
# [Step 1] Progress report entries (Check 3) filled from the numbers above. Paste into the report and label them "dev questions".
b = "all"
def g(system, col, k): return float(compare[(compare["system"] == system) & (compare["lang"] == b) & (compare["k"] == k)][col].iloc[0])
print("Embedding model / system :", BEST)
print("Vector DB / chunks       : FAISS IndexFlatIP,", len(chunks), "chunks, plus BM25")
print("Chunk size / overlap / k : 1000 / 150 / 5 (provisional)")
print("Retrieval metrics (dev, n =", len(answerable), "answerable questions):")
print("  page-hit@1/3/5   =", " / ".join(f"{g(BEST, 'hit', k):.3f}" for k in K_VALUES))
print("  answer-hit@1/3/5 =", " / ".join(f"{g(BEST, 'ans_hit', k):.3f}" for k in K_VALUES))
print(f"  P@5 = {g(BEST, 'precision', 5):.3f}, R@5 = {g(BEST, 'recall', 5):.3f}, MRR@5 = {g(BEST, 'mrr', 5):.3f}")
print("Pipeline status          : Retrieval tested, LLM not yet connected")
print("Unseen query set         : 0 so far (held-out test not written yet)")

Embedding model / system : hybrid:e5_base
Vector DB / chunks       : FAISS IndexFlatIP, 1281 chunks, plus BM25
Chunk size / overlap / k : 1000 / 150 / 5 (provisional)
Retrieval metrics (dev, n = 18 answerable questions):
  page-hit@1/3/5   = 0.722 / 0.944 / 0.944
  answer-hit@1/3/5 = 0.833 / 0.944 / 1.000
  P@5 = 0.378, R@5 = 0.670, MRR@5 = 0.824
Pipeline status          : Retrieval tested, LLM not yet connected
Unseen query set         : 0 so far (held-out test not written yet)


## Refusal finding (write this yourselves)
Use the numbers from Cell 9. Suggested structure, in your own words:
1. What you measured (top-1 cosine score, answerable vs unanswerable, which model, how many questions).
2. What you saw (do the score ranges overlap? which near-miss questions scored highest?).
3. What it means for Week 4 (a score check alone is not enough, so the prompt needs a refusal rule too; which threshold, if any, you will use as a first filter and why).
4. The limit (small number of unanswerable questions; the threshold was picked on the same questions).

## Before Week 4
- [ ] Cell 3a filled in (about 30 answerable, 8-10 Tagalog/Taglish, 6-8 unanswerable) and the whole notebook re-run top to bottom
- [ ] 1500-character test re-run, `over_limit` is 0
- [ ] Dense vs hybrid reported honestly (tie if it is a tie)
- [ ] Refusal finding written
- [ ] `requirements.txt` has all 8 packages
- [ ] Progress report entries updated
- [ ] This chat saved for Section 10 (`ai_transcripts/`)
